In [1]:
import pandas as pd
import cv2
import itertools
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_ann = pd.read_json('dataset/annotation_image_info.json')
df_ann.head(5)

,imgH,source,bbox,source_id,imgW,id
0,640,instagram,"[[62, 266, 325, 639], [224, 219, 564, 636]]",527824347898327040,640,0
1,640,instagram,"[[32, 147, 227, 542], [187, 159, 548, 615]]",527868030970527744,640,1
2,640,instagram,"[[0, 157, 98, 431], [504, 183, 603, 537], [330...",528016560141901824,640,2
3,640,instagram,"[[9, 42, 217, 639], [316, 50, 527, 639], [175,...",528030746167226368,640,3
4,640,instagram,"[[200, 201, 295, 523], [136, 221, 214, 510]]",528034482796761088,640,4


In [3]:
df_rltn = pd.read_json('dataset/relationship.json', orient='index')
df_rltn = df_rltn.sort_index()
df_rltn['id'] = df_rltn.index
df_rltn.head(35)

,1 2,1 3,2 3,2 4,1 4,2 5,1 5,3 4,4 5,4 6,...,4 10,5 11,8 10,5 12,3 12,4 12,3 10,3 11,5 9,id
0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,4.0,4.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
3,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
4,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
5,5.0,5.0,5.0,NaN,NaN,5.0,5.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
6,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
7,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
8,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
9,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9


In [4]:
def image_resize(image, width = None, height = None, inter = cv2.INTER_AREA):
    # initialize the dimensions of the image to be resized and
    # grab the image size
    dim = None
    (h, w) = image.shape[:2]
    if(h==0):
        h=1

    # if both the width and height are None, then return the
    # original image
    if width is None and height is None:
        return image

    # check to see if the width is None
    if width is None:
        # calculate the ratio of the height and construct the
        # dimensions
        r = height / float(h)
        dim = (int(w * r), height)

    # otherwise, the height is None
    else:
        # calculate the ratio of the width and construct the
        # dimensions
        r = width / float(w)
        dim = (width, int(h * r))

    # resize the image
    resized = cv2.resize(image, dim, interpolation = inter)

    # return the resized image
    return resized

In [5]:
def nameIDfixer(id__, total_digits):
    len_id = len(id__)
    newId = ""
    for i in range(total_digits-len_id):
        newId+="0"
        
    return newId+id__

def pairMaker(numOfppl):
    test_list = []
    for i in range(1, numOfppl+1):
        test_list.append(i)
        
    res = [(a, b) for idx, a in enumerate(test_list) for b in test_list[idx + 1:]]
    return res
    
def imgprocessor(_id, numOfp, bbox, nextId, df_row_rltn):
    savePath = "dataset/an/"
    img = cv2.imread("dataset/image/"+nameIDfixer(_id, 5)+".jpg")
    immgs = []
    for index, boxCord in enumerate(bbox):
#         print(index+1, boxCord, savePath)
        crop_img = img[boxCord[1]:boxCord[3], boxCord[0]:boxCord[2]]
        immgs.append(crop_img)
        
    pairCode = pairMaker(numOfp)
    df_ne_an = pd.DataFrame() #dataframe new
    for pair in pairCode:
        img1_ = immgs[list(pair)[0]-1]
        img2_ = immgs[list(pair)[1]-1]
#         print(df_row_rltn[str(list(pair)[0])+" "+str(list(pair)[1])][0])
#         break
        if(pd.isnull(df_row_rltn[str(list(pair)[0])+" "+str(list(pair)[1])].values)):
            continue
#         print("In func: ", str(list(pair)[0])+" "+str(list(pair)[1]), pd.isnull(df_row_rltn[str(list(pair)[0])+" "+str(list(pair)[1])].values))
        rro = {'id_name': str(nextId), 'relation': df_row_rltn[str(list(pair)[0])+" "+str(list(pair)[1])].values[0]}
        df_ne_an = df_ne_an.append(rro, ignore_index=True)
        height1, width1 = img1_.shape[0],img1_.shape[1]
        height2, width2 = img2_.shape[0],img2_.shape[1]
        min_height = max(height1, height2)
#         min_width = min(width1, width2)
        img1_ = image_resize(img1_, height = min_height)
        img2_ = image_resize(img2_, height = min_height)
        im_v = cv2.hconcat([img1_, img2_])
#         print(nextId)
        cv2.imwrite(savePath+str(nextId)+".jpg", im_v)
        nextId+=1
    return nextId, df_ne_an
        
    
    
# imgprocessor("2", 3, [[0, 157, 98, 431], [504, 183, 603, 537], [330, 172, 435, 438]], 1)



In [6]:
indx = 1
df_ne_an = pd.DataFrame()
for index, row in df_ann.iterrows():
#     print("Chk:", row['id'], row['id'] in df_rltn['id'])
#     if(row['id']>10000):
#         break
    if(row['id'] in df_rltn['id']):
        indx, out_ = imgprocessor(str(row['id']), len(row['bbox']), row['bbox'], indx, df_rltn[df_rltn.id==row['id']])
        df_ne_an = pd.concat([df_ne_an,out_], axis=0)
        print("Chk", str(row['id']), row['bbox'])

df_ne_an

Chk 0 [[62, 266, 325, 639], [224, 219, 564, 636]]
Chk 1 [[32, 147, 227, 542], [187, 159, 548, 615]]
Chk 2 [[0, 157, 98, 431], [504, 183, 603, 537], [330, 172, 435, 438]]
Chk 3 [[9, 42, 217, 639], [316, 50, 527, 639], [175, 38, 352, 639]]
Chk 4 [[200, 201, 295, 523], [136, 221, 214, 510]]
Chk 5 [[226, 344, 369, 640], [348, 356, 456, 606], [550, 319, 633, 609], [9, 321, 87, 489], [225, 297, 299, 370], [352, 286, 449, 362]]
Chk 6 [[141, 212, 385, 620], [184, 221, 400, 586], [322, 214, 619, 640]]
Chk 7 [[370, 71, 640, 640], [125, 164, 403, 640], [144, 0, 215, 95]]
Chk 8 [[316, 41, 503, 264], [30, 31, 274, 239]]
Chk 9 [[0, 191, 300, 514], [290, 184, 538, 516]]
Chk 10 [[302, 167, 615, 640], [75, 176, 322, 533]]
Chk 11 [[42, 155, 225, 289], [307, 241, 478, 569], [75, 205, 342, 595]]
Chk 12 [[364, 208, 459, 499], [267, 287, 367, 468]]
Chk 13 [[87, 51, 523, 639], [309, 116, 575, 639]]
Chk 14 [[0, 172, 160, 522], [458, 292, 639, 550], [389, 162, 498, 272], [434, 277, 525, 511], [248, 223, 307, 3

Chk 170 [[308, 24, 427, 293], [183, 22, 325, 281]]
Chk 171 [[184, 256, 298, 372], [78, 278, 148, 364]]
Chk 172 [[420, 0, 500, 121], [351, 0, 428, 95], [247, 20, 372, 171], [260, 112, 349, 251], [213, 112, 277, 211]]
Chk 173 [[152, 51, 298, 146], [252, 40, 395, 139]]
Chk 174 [[112, 50, 376, 372], [62, 94, 184, 362]]
Chk 175 [[236, 31, 470, 321], [7, 46, 201, 331]]
Chk 176 [[36, 134, 150, 313], [137, 126, 314, 342], [228, 93, 375, 287]]
Chk 177 [[107, 30, 214, 277], [231, 32, 362, 265]]
Chk 180 [[364, 163, 466, 318], [463, 55, 500, 193], [305, 78, 388, 333], [163, 109, 252, 333], [0, 43, 135, 333]]
Chk 181 [[455, 238, 499, 329], [95, 100, 273, 324], [241, 111, 413, 328]]
Chk 182 [[132, 111, 220, 304], [323, 109, 397, 255]]
Chk 183 [[42, 28, 147, 289], [151, 16, 344, 333]]
Chk 185 [[35, 91, 211, 292], [168, 21, 490, 338]]
Chk 188 [[305, 40, 446, 314], [44, 25, 238, 297]]
Chk 189 [[0, 61, 90, 294], [149, 94, 296, 290]]
Chk 190 [[328, 48, 469, 340], [31, 62, 235, 387]]
Chk 191 [[246, 174, 3

Chk 306 [[34, 12, 228, 255], [216, 22, 349, 260], [359, 56, 477, 255]]
Chk 307 [[54, 94, 120, 220], [98, 125, 166, 239], [374, 73, 458, 225]]
Chk 308 [[291, 141, 360, 273], [177, 166, 280, 263]]
Chk 309 [[38, 103, 112, 277], [142, 16, 267, 354]]
Chk 310 [[86, 55, 221, 332], [241, 7, 410, 219]]
Chk 311 [[398, 168, 435, 250], [423, 170, 466, 256]]
Chk 312 [[173, 65, 282, 333], [256, 61, 354, 333], [347, 73, 443, 333], [71, 64, 168, 333], [10, 41, 102, 193]]
Chk 314 [[45, 71, 151, 296], [106, 152, 285, 424]]
Chk 317 [[46, 112, 80, 240], [98, 91, 153, 239], [3, 109, 30, 200], [277, 89, 304, 184], [246, 89, 271, 191]]
Chk 318 [[60, 42, 422, 328], [261, 27, 468, 331]]
Chk 320 [[53, 126, 113, 214], [119, 85, 177, 208], [149, 89, 192, 186], [183, 73, 239, 200], [286, 65, 349, 200], [355, 75, 416, 198], [443, 100, 487, 182]]
Chk 321 [[94, 71, 237, 406], [247, 5, 483, 385]]
Chk 322 [[111, 143, 294, 476], [24, 140, 178, 499]]
Chk 323 [[108, 100, 177, 216], [200, 168, 375, 375]]
Chk 324 [[64, 101,

Chk 433 [[64, 56, 197, 311], [204, 35, 380, 333], [379, 39, 491, 333], [337, 47, 408, 333]]
Chk 434 [[11, 22, 115, 391], [83, 49, 333, 500]]
Chk 435 [[109, 102, 190, 369], [5, 0, 289, 402]]
Chk 436 [[82, 163, 213, 495], [126, 170, 200, 283]]
Chk 437 [[169, 38, 318, 333], [18, 103, 124, 323], [317, 100, 458, 317]]
Chk 438 [[228, 74, 305, 273], [306, 31, 375, 332]]
Chk 439 [[92, 38, 186, 332], [236, 43, 345, 332]]
Chk 440 [[335, 48, 407, 315], [263, 61, 387, 333]]
Chk 442 [[95, 117, 288, 493], [5, 135, 124, 499], [27, 134, 129, 301]]
Chk 443 [[143, 73, 235, 211], [212, 83, 335, 229], [351, 78, 427, 229], [419, 88, 492, 235]]
Chk 444 [[13, 48, 315, 332], [173, 17, 469, 332]]
Chk 445 [[113, 152, 232, 329], [257, 46, 400, 332], [238, 128, 264, 210]]
Chk 446 [[123, 106, 256, 331], [202, 133, 263, 290]]
Chk 447 [[247, 58, 342, 332], [131, 42, 247, 332]]
Chk 448 [[51, 103, 220, 492], [195, 112, 333, 477]]
Chk 450 [[37, 32, 234, 268], [153, 47, 451, 316]]
Chk 451 [[98, 49, 256, 498], [255, 56, 

Chk 598 [[287, 23, 462, 303], [65, 46, 158, 312]]
Chk 599 [[89, 23, 261, 329], [243, 6, 399, 317]]
Chk 600 [[241, 126, 287, 235], [161, 143, 212, 238], [323, 132, 374, 230]]
Chk 601 [[34, 25, 270, 331], [161, 27, 478, 294]]
Chk 602 [[59, 0, 212, 332], [267, 120, 424, 328]]
Chk 603 [[241, 79, 355, 332], [141, 87, 233, 332], [0, 116, 178, 332]]
Chk 604 [[63, 66, 156, 334], [201, 61, 309, 310], [347, 51, 468, 322]]
Chk 605 [[0, 153, 148, 375], [129, 150, 335, 375], [299, 120, 375, 230], [418, 117, 487, 253]]
Chk 606 [[205, 44, 428, 333], [176, 42, 412, 333]]
Chk 607 [[72, 36, 302, 374], [293, 0, 462, 299]]
Chk 608 [[65, 142, 200, 485], [164, 28, 421, 485], [146, 211, 268, 483]]
Chk 609 [[376, 73, 463, 232], [244, 84, 335, 226], [119, 76, 184, 239], [4, 69, 71, 232]]
Chk 611 [[298, 77, 385, 334], [127, 65, 231, 320]]
Chk 612 [[0, 0, 217, 500], [102, 0, 332, 500]]
Chk 613 [[56, 169, 168, 483], [126, 239, 210, 472], [257, 233, 332, 458], [172, 141, 263, 493]]
Chk 614 [[233, 75, 398, 238], [5

...

Chk 15303 [[29, 74, 90, 297], [159, 59, 222, 296], [353, 175, 561, 421], [691, 118, 741, 270], [544, 104, 601, 183], [110, 82, 157, 257], [436, 193, 597, 374], [307, 111, 368, 214], [626, 55, 686, 281]]
Chk 15304 [[88, 22, 446, 499], [227, 165, 404, 480]]
Chk 15305 [[148, 214, 419, 757], [141, 19, 442, 799]]
Chk 15306 [[90, 331, 520, 586], [408, 98, 595, 592], [353, 33, 527, 305]]
Chk 15308 [[224, 104, 546, 534], [499, 270, 799, 534]]
Chk 15309 [[273, 94, 546, 799], [519, 272, 597, 387], [113, 154, 331, 694], [39, 221, 93, 360], [0, 185, 44, 334], [51, 210, 140, 375]]
Chk 15310 [[251, 280, 419, 518], [61, 147, 253, 524], [325, 29, 743, 524], [0, 174, 72, 518], [8, 1, 116, 322], [679, 159, 798, 524], [259, 36, 472, 524]]
Chk 15311 [[408, 44, 793, 507], [6, 106, 388, 516], [137, 28, 671, 516]]
Chk 15312 [[0, 163, 42, 301], [65, 44, 210, 383], [20, 76, 332, 493]]
Chk 15313 [[364, 4, 765, 633], [205, 7, 472, 633]]
Chk 15314 [[0, 152, 175, 490], [295, 84, 374, 236]]
Chk 15316 [[155, 141, 36

Chk 15445 [[248, 82, 409, 486], [59, 136, 158, 523], [397, 63, 529, 571]]
Chk 15446 [[0, 11, 313, 532], [330, 54, 794, 523]]
Chk 15447 [[303, 122, 589, 406], [542, 98, 799, 501], [18, 101, 334, 344]]
Chk 15448 [[273, 28, 645, 610], [103, 256, 502, 625]]
Chk 15449 [[45, 74, 358, 599], [443, 4, 766, 599]]
Chk 15450 [[203, 2, 662, 648], [442, 45, 669, 580], [173, 62, 481, 492]]
Chk 15451 [[105, 48, 338, 570], [379, 82, 621, 562]]
Chk 15453 [[174, 28, 291, 333], [229, 9, 303, 330], [441, 116, 493, 229]]
Chk 15454 [[191, 54, 398, 600], [261, 92, 800, 600]]
Chk 15455 [[0, 72, 346, 373], [196, 51, 481, 368], [380, 43, 495, 366], [198, 74, 333, 330]]
Chk 15456 [[106, 184, 422, 600], [275, 190, 501, 600]]
Chk 15457 [[140, 44, 284, 272], [27, 68, 152, 332]]
Chk 15458 [[15, 105, 377, 680], [326, 94, 552, 650], [456, 119, 727, 679], [723, 125, 784, 347], [0, 242, 42, 397], [3, 104, 46, 319], [134, 93, 181, 165]]
Chk 15459 [[180, 30, 548, 522], [449, 144, 783, 532]]
Chk 15460 [[3, 58, 319, 556], [1

Chk 15588 [[384, 92, 721, 641], [60, 49, 401, 640], [358, 242, 409, 392], [294, 214, 353, 349]]
Chk 15589 [[569, 188, 787, 532], [8, 135, 185, 532], [230, 70, 549, 521], [130, 136, 293, 519], [284, 50, 433, 363]]
Chk 15590 [[706, 196, 790, 408], [238, 86, 434, 527], [553, 171, 692, 520], [512, 165, 586, 386], [339, 189, 549, 496], [584, 115, 646, 217]]
Chk 15591 [[29, 35, 191, 295], [303, 124, 370, 247], [189, 140, 267, 223]]
Chk 15592 [[314, 10, 403, 241], [412, 94, 470, 156], [0, 139, 163, 335], [174, 115, 347, 278]]
Chk 15593 [[4, 0, 111, 170], [89, 110, 262, 333], [108, 269, 324, 481], [190, 488, 448, 723], [260, 580, 582, 800]]
Chk 15594 [[146, 16, 320, 460], [306, 66, 516, 400], [579, 17, 746, 360]]
Chk 15595 [[8, 79, 329, 578], [499, 67, 799, 596], [338, 100, 636, 538], [244, 141, 391, 487]]
Chk 15596 [[54, 110, 301, 428], [382, 119, 570, 406], [534, 95, 786, 449], [704, 72, 800, 188], [465, 53, 575, 162]]
Chk 15597 [[563, 53, 775, 272], [3, 8, 399, 532], [377, 50, 625, 388]]
Ch

Chk 15715 [[432, 40, 800, 589], [356, 212, 589, 452], [172, 218, 429, 433]]
Chk 15716 [[229, 107, 467, 529], [404, 84, 669, 533]]
Chk 15717 [[101, 5, 446, 515], [457, 34, 728, 525], [218, 49, 285, 348]]
Chk 15718 [[0, 64, 160, 533], [583, 121, 770, 533], [110, 131, 350, 533], [397, 54, 655, 533], [0, 52, 46, 136], [99, 47, 143, 127], [57, 45, 102, 112]]
Chk 15719 [[484, 40, 800, 532], [204, 197, 587, 532], [54, 123, 272, 531], [370, 277, 414, 394]]
Chk 15720 [[0, 33, 364, 599], [498, 81, 799, 513]]
Chk 15721 [[0, 68, 114, 572], [525, 76, 740, 576], [64, 76, 398, 576]]
Chk 15722 [[281, 146, 520, 484], [29, 110, 485, 528], [499, 137, 763, 463]]
Chk 15724 [[400, 124, 797, 527], [34, 53, 490, 535]]
Chk 15725 [[103, 273, 456, 530], [303, 245, 507, 530]]
Chk 15726 [[141, 131, 367, 372], [256, 70, 389, 374]]
Chk 15727 [[188, 58, 301, 325], [329, 46, 449, 327]]
Chk 15728 [[501, 103, 793, 568], [19, 80, 371, 580]]
Chk 15729 [[299, 23, 473, 188], [52, 41, 391, 332], [1, 23, 224, 254]]
Chk 15730 

error: OpenCV(4.7.0) /Users/runner/miniforge3/conda-bld/libopencv_1675730072788/work/modules/imgproc/src/resize.cpp:4062: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [ ]:
df_ne_an.to_csv('dataset/annotations1.csv',index=False)
df_ne_an

In [ ]:
# img = cv2.imread("dataset/image/00000.jpg")

# cv2.imshow('sample image',img)

# cv2.imwrite(, img)

In [ ]:
# import cv2
# import matplotlib.pyplot as plt

# image = cv2.imread('dataset/image/00001.jpg')
# indx = [[539, 234, 591, 424], [585, 168, 637, 435], [240, 79, 383, 219], [153, 111, 215, 221], [7, 165, 72, 312]]
# indx = indx[1]

# height, width, channels = image.shape
# start_point = (indx[0],indx[1])
# end_point = (indx[2], indx[3])
# color = (0,0,255)
# thickness = 5

# image = cv2.rectangle(image, start_point, end_point, color, thickness)
# # cv2.imshow('Rectangle',image)

# image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
# plt.imshow(image)
# plt.show()